In [16]:
from pathlib import Path
import sys
import time

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import (
    build_logistic_regression_pipeline,
    build_svm_pipeline,
    get_logistic_regression_param_grid,
    get_svm_param_grid
)

print("Project root:", PROJECT_ROOT)

Project root: /home/jadkandah/progressSoft_internship/phase-1-machine-learning-nlp/assignment


In [17]:
TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_clean.csv"
)

VALIDATION_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "validation_clean.csv"
)

train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Training shape: (69384, 4)
Validation shape: (1000, 4)


In [18]:
X_train = train_df["text"]
y_train = train_df["sentiment"]

X_validation = validation_df["text"]
y_validation = validation_df["sentiment"]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_validation))
print("\nTraining label distribution:")
print(y_train.value_counts())

Training samples: 69384
Validation samples: 1000

Training label distribution:
sentiment
Negative      21157
Positive      19068
Neutral       16978
Irrelevant    12181
Name: count, dtype: int64


In [19]:
cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

# Logistic Regression Tuning

In [20]:
logistic_param_grid = get_logistic_regression_param_grid()


In [21]:
logistic_search = GridSearchCV(
    estimator=build_logistic_regression_pipeline(),
    param_grid=logistic_param_grid,
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    refit=True,
)

In [22]:
start_time = time.perf_counter()

logistic_search.fit(X_train, y_train)

logistic_search_time = (
    time.perf_counter() - start_time
)

print(
    f"Logistic Regression search time: "
    f"{logistic_search_time:.2f} seconds"
)

print(
    "Best parameters:",
    logistic_search.best_params_,
)

print(
    "Best cross-validation macro F1:",
    logistic_search.best_score_,
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


[CV] END classifier__C=0.1, classifier__class_weight=balanced; total time=  39.6s
[CV] END classifier__C=0.1, classifier__class_weight=balanced; total time=  39.8s
[CV] END classifier__C=0.1, classifier__class_weight=balanced; total time=  39.9s
[CV] END ...classifier__C=0.1, classifier__class_weight=None; total time=  43.1s
[CV] END ...classifier__C=0.1, classifier__class_weight=None; total time=  44.1s
[CV] END classifier__C=0.5, classifier__class_weight=balanced; total time=  46.1s
[CV] END ...classifier__C=0.1, classifier__class_weight=None; total time=  48.2s
[CV] END classifier__C=0.5, classifier__class_weight=balanced; total time=  53.7s
[CV] END ...classifier__C=0.5, classifier__class_weight=None; total time=  54.1s
[CV] END ...classifier__C=0.5, classifier__class_weight=None; total time=  55.1s
[CV] END classifier__C=0.5, classifier__class_weight=balanced; total time=  56.1s
[CV] END ...classifier__C=0.5, classifier__class_weight=None; total time= 1.0min
[CV] END ...classifier

In [23]:
logistic_cv_results = pd.DataFrame(
    logistic_search.cv_results_
)

logistic_cv_summary = (
    logistic_cv_results[
        [
            "params",
            "mean_train_score",
            "std_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

display(logistic_cv_summary)

,params,mean_train_score,std_train_score,mean_test_score,std_test_score,rank_test_score,mean_fit_time
0,"{'classifier__C': 5.0, 'classifier__class_weig...",0.986645,0.000593,0.893646,0.001410,1,47.973337
1,"{'classifier__C': 5.0, 'classifier__class_weig...",0.986413,0.000775,0.891314,0.001056,2,53.614146
2,"{'classifier__C': 2.0, 'classifier__class_weig...",0.969309,0.000928,0.866501,0.000150,3,68.102623
3,"{'classifier__C': 2.0, 'classifier__class_weig...",0.968595,0.000366,0.861568,0.001777,4,76.270546
4,"{'classifier__C': 1.0, 'classifier__class_weig...",0.941560,0.000543,0.830300,0.001761,5,66.176056
5,"{'classifier__C': 1.0, 'classifier__class_weig...",0.936379,0.001098,0.818874,0.002002,6,61.993111
6,"{'classifier__C': 0.5, 'classifier__class_weig...",0.892080,0.001026,0.781554,0.000389,7,43.043008
7,"{'classifier__C': 0.5, 'classifier__class_weig...",0.874643,0.002179,0.757544,0.004392,8,47.288787
8,"{'classifier__C': 0.1, 'classifier__class_weig...",0.737830,0.000720,0.663007,0.003966,9,29.621142
9,"{'classifier__C': 0.1, 'classifier__class_weig...",0.619178,0.003053,0.563018,0.004576,10,34.787389


# Linear SVM Tuning

In [24]:
svm_param_grid = get_svm_param_grid()

svm_search = GridSearchCV(
    estimator=build_svm_pipeline(),
    param_grid=svm_param_grid,
    scoring="f1_macro",
    cv=cv_strategy,
    n_jobs=-1,
    verbose=2,
    return_train_score=True,
    refit=True,
)

In [25]:
start_time = time.perf_counter()

svm_search.fit(X_train, y_train)

svm_search_time = (
    time.perf_counter() - start_time
)

print(
    f"SVM search time: "
    f"{svm_search_time:.2f} seconds"
)

print(
    "Best parameters:",
    svm_search.best_params_,
)

print(
    "Best cross-validation macro F1:",
    svm_search.best_score_,
)

Fitting 3 folds for each of 14 candidates, totalling 42 fits
[CV] END ..classifier__C=0.01, classifier__class_weight=None; total time=  26.9s
[CV] END ..classifier__C=0.01, classifier__class_weight=None; total time=  25.8s
[CV] END classifier__C=0.01, classifier__class_weight=balanced; total time=  27.1s
[CV] END ..classifier__C=0.01, classifier__class_weight=None; total time=  28.6s
[CV] END classifier__C=0.05, classifier__class_weight=balanced; total time=  29.1s
[CV] END ..classifier__C=0.05, classifier__class_weight=None; total time=  29.7s
[CV] END classifier__C=0.01, classifier__class_weight=balanced; total time=  29.6s
[CV] END classifier__C=0.01, classifier__class_weight=balanced; total time=  30.0s
[CV] END classifier__C=0.05, classifier__class_weight=balanced; total time=  29.7s
[CV] END ..classifier__C=0.05, classifier__class_weight=None; total time=  28.5s
[CV] END ..classifier__C=0.05, classifier__class_weight=None; total time=  29.9s
[CV] END classifier__C=0.05, classifie

In [26]:
svm_cv_results = pd.DataFrame(
    svm_search.cv_results_
)

svm_cv_summary = (
    svm_cv_results[
        [
            "params",
            "mean_train_score",
            "std_train_score",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
            "mean_fit_time",
        ]
    ]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

display(svm_cv_summary)

,params,mean_train_score,std_train_score,mean_test_score,std_test_score,rank_test_score,mean_fit_time
0,"{'classifier__C': 2.0, 'classifier__class_weig...",0.991749,0.000432,0.920125,0.000303,1,30.420307
1,"{'classifier__C': 5.0, 'classifier__class_weig...",0.992802,0.000458,0.920090,0.000459,2,26.963719
2,"{'classifier__C': 2.0, 'classifier__class_weig...",0.991631,0.000345,0.920084,0.000239,3,29.724173
3,"{'classifier__C': 5.0, 'classifier__class_weig...",0.992757,0.000332,0.919956,0.000256,4,28.090281
4,"{'classifier__C': 1.0, 'classifier__class_weig...",0.989299,0.000274,0.914456,0.000100,5,25.870127
5,"{'classifier__C': 1.0, 'classifier__class_weig...",0.989255,0.000351,0.913409,0.000262,6,22.925110
6,"{'classifier__C': 0.5, 'classifier__class_weig...",0.983648,0.000429,0.899695,0.000409,7,23.229791
7,"{'classifier__C': 0.5, 'classifier__class_weig...",0.983232,0.000357,0.898104,0.000619,8,22.665375
8,"{'classifier__C': 0.1, 'classifier__class_weig...",0.923707,0.000709,0.814075,0.001565,9,20.879115
9,"{'classifier__C': 0.1, 'classifier__class_weig...",0.913436,0.000987,0.797502,0.001436,10,19.967714


In [27]:
def evaluate_model(model,X,y,model_name: str):

    start_time = time.perf_counter()

    predictions = model.predict(X)

    prediction_time = (
        time.perf_counter() - start_time
    )

    accuracy = accuracy_score(
        y,
        predictions,
    )

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _,
    ) = precision_recall_fscore_support(
        y,
        predictions,
        average="macro",
        zero_division=0,
    )

    (
        weighted_precision,
        weighted_recall,
        weighted_f1,
        _,
    ) = precision_recall_fscore_support(
        y,
        predictions,
        average="weighted",
        zero_division=0,
    )

    metrics = pd.DataFrame(
        {
            "model": [model_name],
            "accuracy": [accuracy],
            "macro_precision": [
                macro_precision
            ],
            "macro_recall": [macro_recall],
            "macro_f1": [macro_f1],
            "weighted_precision": [
                weighted_precision
            ],
            "weighted_recall": [
                weighted_recall
            ],
            "weighted_f1": [weighted_f1],
            "prediction_time_seconds": [
                prediction_time
            ],
        }
    )

    predictions_df = pd.DataFrame(
        {
            "actual": y.to_numpy(),
            "predicted": predictions,
        }
    )

    return metrics, predictions_df

In [28]:
tuned_logistic_metrics, tuned_logistic_predictions = (
    evaluate_model(
        model=logistic_search.best_estimator_,
        X=X_validation,
        y=y_validation,
        model_name="Tuned Logistic Regression",
    )
)

display(tuned_logistic_metrics)

print(
    classification_report(
        y_validation,
        tuned_logistic_predictions["predicted"],
        zero_division=0,
    )
)

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,prediction_time_seconds
0,Tuned Logistic Regression,0.975,0.976067,0.975325,0.975568,0.975403,0.975,0.975059,0.126699


              precision    recall  f1-score   support

  Irrelevant       0.98      0.98      0.98       172
    Negative       0.98      0.98      0.98       266
     Neutral       0.99      0.96      0.98       285
    Positive       0.95      0.98      0.96       277

    accuracy                           0.97      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.97      0.98      1000



In [29]:
tuned_svm_metrics, tuned_svm_predictions = (
    evaluate_model(
        model=svm_search.best_estimator_,
        X=X_validation,
        y=y_validation,
        model_name="Tuned Linear SVM",
    )
)

display(tuned_svm_metrics)

print(
    classification_report(
        y_validation,
        tuned_svm_predictions["predicted"],
        zero_division=0,
    )
)

,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,prediction_time_seconds
0,Tuned Linear SVM,0.98,0.979784,0.980463,0.980008,0.980244,0.98,0.979993,0.117617


              precision    recall  f1-score   support

  Irrelevant       0.98      0.98      0.98       172
    Negative       0.98      0.99      0.98       266
     Neutral       1.00      0.96      0.98       285
    Positive       0.97      0.99      0.98       277

    accuracy                           0.98      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.98      0.98      0.98      1000



In [30]:
model_comparison = pd.DataFrame(
    [
        {
            "model": "Baseline Logistic Regression",
            "accuracy": 0.9680,
            "macro_f1": 0.967366,
            "weighted_f1": 0.967961,
            "training_or_search_time_seconds": 43.104806,
        },
        {
            "model": "Baseline Linear SVM",
            "accuracy": 0.9770,
            "macro_f1": 0.977047,
            "weighted_f1": 0.976995,
            "training_or_search_time_seconds": 9.383048,
        },
        {
            "model": "Tuned Logistic Regression",
            "accuracy": (
                tuned_logistic_metrics
                .loc[0, "accuracy"]
            ),
            "macro_f1": (
                tuned_logistic_metrics
                .loc[0, "macro_f1"]
            ),
            "weighted_f1": (
                tuned_logistic_metrics
                .loc[0, "weighted_f1"]
            ),
            "training_or_search_time_seconds": (
                logistic_search_time
            ),
        },
        {
            "model": "Tuned Linear SVM",
            "accuracy": (
                tuned_svm_metrics
                .loc[0, "accuracy"]
            ),
            "macro_f1": (
                tuned_svm_metrics
                .loc[0, "macro_f1"]
            ),
            "weighted_f1": (
                tuned_svm_metrics
                .loc[0, "weighted_f1"]
            ),
            "training_or_search_time_seconds": (
                svm_search_time
            ),
        },
    ]
)

model_comparison = model_comparison.sort_values(
    by="macro_f1",
    ascending=False,
).reset_index(drop=True)

display(model_comparison)

,model,accuracy,macro_f1,weighted_f1,training_or_search_time_seconds
0,Tuned Linear SVM,0.980,0.980008,0.979993,198.171309
1,Baseline Linear SVM,0.977,0.977047,0.976995,9.383048
2,Tuned Logistic Regression,0.975,0.975568,0.975059,272.951369
3,Baseline Logistic Regression,0.968,0.967366,0.967961,43.104806


In [31]:
RESULTS_DIR = (
    PROJECT_ROOT
    / "reports"
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [32]:
logistic_cv_summary.to_csv(
    RESULTS_DIR
    / "logistic_regression_tuning_results.csv",
    index=False,
)

svm_cv_summary.to_csv(
    RESULTS_DIR
    / "svm_tuning_results.csv",
    index=False,
)


model_comparison.to_csv(
    RESULTS_DIR
    / "tuned_model_comparison.csv",
    index=False,
)

In [33]:
best_parameters = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "best_parameters": str(
                logistic_search.best_params_
            ),
            "best_cv_macro_f1": (
                logistic_search.best_score_
            ),
        },
        {
            "model": "Linear SVM",
            "best_parameters": str(
                svm_search.best_params_
            ),
            "best_cv_macro_f1": (
                svm_search.best_score_
            ),
        },
    ]
)

best_parameters.to_csv(
    RESULTS_DIR
    / "best_hyperparameters.csv",
    index=False,
)

display(best_parameters)

,model,best_parameters,best_cv_macro_f1
0,Logistic Regression,"{'classifier__C': 5.0, 'classifier__class_weig...",0.893646
1,Linear SVM,"{'classifier__C': 2.0, 'classifier__class_weig...",0.920125
